Setup

In [2]:
from pathlib import Path
import xml.etree.ElementTree as ET
import numpy as np
import cv2
import matplotlib.pyplot as plt
import random

XML_PATH = Path(
    "data/landmarks/landmarks_P001_P030/annotations.xml"
)

IMAGE_DIR = Path(
    "data/selected_photos"
)

tree = ET.parse(XML_PATH)
root = tree.getroot()

images = root.findall(".//image")

print("Number of images:", len(images))

Number of images: 30


Landmarks exportation

In [3]:
landmark_labels = [
    "left_eye_inner_corner",
    "left_eye_outer_corner",
    "left_white_point",
    "right_eye_inner_corner",
    "right_eye_outer_corner",
    "right_white_point",
]

landmarks = {}

for image in images:
    image_name = image.attrib["name"]

    landmarks[image_name] = {}

    for point in image.findall("points"):
        label = point.attrib["label"]

        if label in landmark_labels:
            x, y = map(
                float,
                point.attrib["points"].split(",")
            )

            landmarks[image_name][label] = (x, y)

print("Landmarks loaded for:", len(landmarks), "images")

Landmarks loaded for: 30 images


Masks exportation

In [4]:
iris_labels = [
    "left_iris_mask",
    "right_iris_mask"
]

iris_masks = {}

for image in images:
    image_name = image.attrib["name"]

    iris_masks[image_name] = {}

    for polygon in image.findall("polygon"):
        label = polygon.attrib["label"]

        if label in iris_labels:
            points_str = polygon.attrib["points"]

            points = []

            for p in points_str.split(";"):
                x, y = map(float, p.split(","))
                points.append((x, y))

            iris_masks[image_name][label] = points

print("Masks loaded for:", len(iris_masks), "images")

Masks loaded for: 30 images


Split 30 images randomly for train, validation and test

In [5]:
image_names = list(landmarks.keys())

random.seed(42)
random.shuffle(image_names)

train_names = image_names[:21]
val_names = image_names[21:27]
test_names = image_names[27:]

print("Train:", len(train_names))
print("Val:", len(val_names))
print("Test:", len(test_names))

Train: 21
Val: 6
Test: 3


Defining points

In [6]:
left_keypoints = [
    "left_eye_inner_corner",
    "left_eye_outer_corner",
    "left_white_point"
]

right_keypoints = [
    "right_eye_inner_corner",
    "right_eye_outer_corner",
    "right_white_point"
]

Bounding box function

In [7]:
def create_eye_bbox(
    landmarks_dict,
    iris_dict,
    keypoint_labels,
    iris_label,
    padding_ratio=0.15
):
    points = [
        landmarks_dict[label]
        for label in keypoint_labels
    ]

    points.extend(iris_dict[iris_label])

    points = np.array(points, dtype=np.float32)

    x_min = points[:, 0].min()
    x_max = points[:, 0].max()
    y_min = points[:, 1].min()
    y_max = points[:, 1].max()

    width = x_max - x_min
    height = y_max - y_min

    x_padding = width * padding_ratio
    y_padding = height * padding_ratio

    x_min -= x_padding
    x_max += x_padding
    y_min -= y_padding
    y_max += y_padding

    return [x_min, y_min, x_max, y_max]

Turning bounding box into YOLO format

In [8]:
def bbox_to_yolo(bbox, img_width, img_height):

    x1, y1, x2, y2 = bbox

    x1 = max(0, min(x1, img_width))
    x2 = max(0, min(x2, img_width))

    y1 = max(0, min(y1, img_height))
    y2 = max(0, min(y2, img_height))

    x_center = (x1 + x2) / 2
    y_center = (y1 + y2) / 2

    width = x2 - x1
    height = y2 - y1

    return [
        x_center / img_width,
        y_center / img_height,
        width / img_width,
        height / img_height
    ]

Creating label function

In [9]:
def create_yolo_label(image_name):

    img_path = IMAGE_DIR / image_name

    img = cv2.imread(str(img_path))

    if img is None:
        raise FileNotFoundError(
            f"Could not load image: {img_path}"
        )

    img_height, img_width = img.shape[:2]

    image_landmarks = landmarks[image_name]
    image_masks = iris_masks[image_name]

    lines = []


    # LEFT EYE ---------------------

    left_bbox = create_eye_bbox(
        image_landmarks,
        image_masks,
        left_keypoints,
        "left_iris_mask"
    )

    left_bbox_yolo = bbox_to_yolo(
        left_bbox,
        img_width,
        img_height
    )

    left_points = [
        image_landmarks["left_eye_inner_corner"],
        image_landmarks["left_eye_outer_corner"],
        image_landmarks["left_white_point"]
    ]

    left_kps = []

    for x, y in left_points:
        left_kps.extend([
            x / img_width,
            y / img_height,
            2
        ])

    left_line = [
        0,
        *left_bbox_yolo,
        *left_kps
    ]

    lines.append(left_line)

    # RIGHT EYE ---------------------

    right_bbox = create_eye_bbox(
        image_landmarks,
        image_masks,
        right_keypoints,
        "right_iris_mask"
    )

    right_bbox_yolo = bbox_to_yolo(
        right_bbox,
        img_width,
        img_height
    )

    right_points = [
        image_landmarks["right_eye_inner_corner"],
        image_landmarks["right_eye_outer_corner"],
        image_landmarks["right_white_point"]
    ]

    right_kps = []

    for x, y in right_points:
        right_kps.extend([
            x / img_width,
            y / img_height,
            2
        ])

    right_line = [
        0,
        *right_bbox_yolo,
        *right_kps
    ]

    lines.append(right_line)

    return lines

Creating the dataset

In [15]:
import shutil

YOLO_DIR = Path("data/yolo_pose")
IMAGE_DIR = Path("data/selected_photos")

splits = {
    "train": train_names,
    "val": val_names,
    "test": test_names,
}

for split, names in splits.items():

    for image_name in names:

        src = IMAGE_DIR / image_name
        dst = YOLO_DIR / "images" / split / image_name

        shutil.copy2(src, dst)

        # Create YOLO label

        lines = create_yolo_label(image_name)

        label_path = (
            YOLO_DIR
            / "labels"
            / split
            / f"{Path(image_name).stem}.txt"
        )

        with open(label_path, "w") as f:

            for line in lines:

                line_str = " ".join(
                    str(float(x)) if i > 0 else str(int(x))
                    for i, x in enumerate(line)
                )

                f.write(line_str + "\n")

print("Dataset created successfully!")

Dataset created successfully!


In [16]:
for split in ["train", "val", "test"]:

    image_count = len(
        list((YOLO_DIR / "images" / split).glob("*"))
    )

    label_count = len(
        list((YOLO_DIR / "labels" / split).glob("*.txt"))
    )

    print(
        f"{split}: "
        f"{image_count} images, "
        f"{label_count} labels"
    )

train: 21 images, 21 labels
val: 6 images, 6 labels
test: 3 images, 3 labels


Creating data.yaml

In [17]:
yaml_content = """
path: data/yolo_pose

train: images/train
val: images/val
test: images/test

kpt_shape: [3, 3]

names:
  0: eye
"""

yaml_path = YOLO_DIR / "data.yaml"

with open(yaml_path, "w") as f:
    f.write(yaml_content.strip())

print(yaml_path)

data\yolo_pose\data.yaml


Train using a pretrained model

In [19]:
from ultralytics import YOLO

model = YOLO("yolo11n-pose.pt")

In [21]:
results = model.train(
    data="data/yolo_pose/data.yaml",

    epochs=100,

    imgsz=640,

    batch=4,

    patience=20,

    device="cpu",

    project="runs/eye_strabismus",
    name="yolo_pose_test",

    pretrained=True,

    workers=2,

    plots=True
)

Ultralytics 8.4.120  Python-3.14.6 torch-2.13.0+cpu CPU (12th Gen Intel Core i7-1255U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data/yolo_pose/data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-pose.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo_pose_test-2, nbs=64, 

KeyboardInterrupt: 

In [20]:
metrics = model.val(
    data="data/yolo_pose/data.yaml",
    split="test"
)

print(metrics)

Ultralytics 8.4.120  Python-3.14.6 torch-2.13.0+cpu CPU (12th Gen Intel Core i7-1255U)
YOLO11n-pose summary (fused): 109 layers, 2,866,468 parameters, 0 gradients, 7.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 89.52.8 MB/s, size: 2122.0 KB)
val: Scanning C:\Users\Asus\Desktop\Files\EyeDerivationProject\data\yolo_pose\labels\test.cache... 3 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 3/3 273.5Kit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Pose(P          R      mAP50  mAP50-95): 0% ──────────── 0/1  1.2s


RuntimeError: shape '[-1, 3, 3]' is invalid for input of size 204

In [22]:
best_model = YOLO(
    "runs/pose/runs/eye_strabismus/yolo_pose_test/weights/best.pt"
)

In [23]:
test_results = best_model.predict(
    source="data/yolo_pose/images/test",
    imgsz=640,
    conf=0.25,
    save=True,
    save_txt=True
)


image 1/3 c:\Users\Asus\Desktop\Files\EyeDerivationProject\data\yolo_pose\images\test\P001_IMG_7116.JPG: 640x480 2 eyes, 77.9ms
image 2/3 c:\Users\Asus\Desktop\Files\EyeDerivationProject\data\yolo_pose\images\test\P004_IMG_7074.JPG: 640x480 2 eyes, 63.2ms
image 3/3 c:\Users\Asus\Desktop\Files\EyeDerivationProject\data\yolo_pose\images\test\P021_IMG_7532.JPG: 640x480 2 eyes, 65.6ms
Speed: 4.1ms preprocess, 68.9ms inference, 1.1ms postprocess per image at shape (1, 3, 640, 480)
Results saved to C:\Users\Asus\Desktop\Files\EyeDerivationProject\runs\pose\predict-2
3 labels saved to C:\Users\Asus\Desktop\Files\EyeDerivationProject\runs\pose\predict-2\labels


In [24]:
import numpy as np
from pathlib import Path

KEYPOINT_ORDER = [
    "inner",
    "outer",
    "white"
]

GT_KEYPOINT_LABELS = [
    ["left_eye_inner_corner",
     "left_eye_outer_corner",
     "left_white_point"],

    ["right_eye_inner_corner",
     "right_eye_outer_corner",
     "right_white_point"]
]

In [25]:
def calculate_keypoint_errors(pred_points, gt_points):
    """
    Calculate Euclidean distance between
    predicted and ground-truth keypoints.
    """

    pred_points = np.array(pred_points, dtype=np.float32)
    gt_points = np.array(gt_points, dtype=np.float32)

    distances = np.sqrt(
        np.sum((pred_points - gt_points) ** 2, axis=1)
    )

    return distances

In [26]:
def bbox_iou(box1, box2):
    """
    box format:
    [x1, y1, x2, y2]
    """

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])

    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter_w = max(0, x2 - x1)
    inter_h = max(0, y2 - y1)

    intersection = inter_w * inter_h

    area1 = (
        max(0, box1[2] - box1[0]) *
        max(0, box1[3] - box1[1])
    )

    area2 = (
        max(0, box2[2] - box2[0]) *
        max(0, box2[3] - box2[1])
    )

    union = area1 + area2 - intersection

    if union == 0:
        return 0

    return intersection / union

In [27]:
all_errors = []

for result in test_results:

    image_name = Path(result.path).name

    print("\nImage:", image_name)

    # -----------------------------
    # Ground Truth
    # -----------------------------

    image_landmarks = landmarks[image_name]

    gt_left = [
        image_landmarks["left_eye_inner_corner"],
        image_landmarks["left_eye_outer_corner"],
        image_landmarks["left_white_point"]
    ]

    gt_right = [
        image_landmarks["right_eye_inner_corner"],
        image_landmarks["right_eye_outer_corner"],
        image_landmarks["right_white_point"]
    ]

    gt_eyes = [
        gt_left,
        gt_right
    ]

    # Predictions

    if result.keypoints is None:
        print("No keypoints predicted.")
        continue

    pred_points = result.keypoints.xy.cpu().numpy()

    print("Number of predictions:", len(pred_points))

    # Match predictions to GT

    if len(pred_points) != 2:
        print("WARNING: Expected 2 eyes, got", len(pred_points))
        continue

    # YOLO prediction order may differ, Try both possible assignments..

    errors_same = (
        calculate_keypoint_errors(
            pred_points[0],
            gt_eyes[0]
        ).sum()
        +
        calculate_keypoint_errors(
            pred_points[1],
            gt_eyes[1]
        ).sum()
    )

    errors_swapped = (
        calculate_keypoint_errors(
            pred_points[0],
            gt_eyes[1]
        ).sum()
        +
        calculate_keypoint_errors(
            pred_points[1],
            gt_eyes[0]
        ).sum()
    )

    if errors_same <= errors_swapped:
        assignments = [
            (0, 0),
            (1, 1)
        ]
    else:
        assignments = [
            (0, 1),
            (1, 0)
        ]

    # Calculate errors

    for pred_idx, gt_idx in assignments:

        errors = calculate_keypoint_errors(
            pred_points[pred_idx],
            gt_eyes[gt_idx]
        )

        print(
            "Eye:",
            "Left" if gt_idx == 0 else "Right"
        )

        for kp_name, error in zip(
            KEYPOINT_ORDER,
            errors
        ):
            print(
                f"  {kp_name}: {error:.2f} px"
            )

            all_errors.append({
                "image": image_name,
                "eye": "left" if gt_idx == 0 else "right",
                "keypoint": kp_name,
                "error_px": float(error)
            })


Image: P001_IMG_7116.JPG
Number of predictions: 2
Eye: Right
  inner: 10.80 px
  outer: 24.25 px
  white: 10.33 px
Eye: Left
  inner: 40.62 px
  outer: 45.00 px
  white: 26.97 px

Image: P004_IMG_7074.JPG
Number of predictions: 2
Eye: Left
  inner: 19.77 px
  outer: 17.10 px
  white: 37.34 px
Eye: Right
  inner: 9.96 px
  outer: 29.02 px
  white: 31.42 px

Image: P021_IMG_7532.JPG
Number of predictions: 2
Eye: Right
  inner: 45.69 px
  outer: 29.61 px
  white: 3.50 px
Eye: Left
  inner: 31.91 px
  outer: 4.89 px
  white: 45.77 px


In [28]:
import pandas as pd

errors_df = pd.DataFrame(all_errors)

print(errors_df)

                image    eye keypoint   error_px
0   P001_IMG_7116.JPG  right    inner  10.802475
1   P001_IMG_7116.JPG  right    outer  24.249031
2   P001_IMG_7116.JPG  right    white  10.325572
3   P001_IMG_7116.JPG   left    inner  40.616623
4   P001_IMG_7116.JPG   left    outer  44.999199
5   P001_IMG_7116.JPG   left    white  26.966303
6   P004_IMG_7074.JPG   left    inner  19.769747
7   P004_IMG_7074.JPG   left    outer  17.099968
8   P004_IMG_7074.JPG   left    white  37.338562
9   P004_IMG_7074.JPG  right    inner   9.963189
10  P004_IMG_7074.JPG  right    outer  29.017591
11  P004_IMG_7074.JPG  right    white  31.423574
12  P021_IMG_7532.JPG  right    inner  45.685276
13  P021_IMG_7532.JPG  right    outer  29.611504
14  P021_IMG_7532.JPG  right    white   3.501449
15  P021_IMG_7532.JPG   left    inner  31.906706
16  P021_IMG_7532.JPG   left    outer   4.891379
17  P021_IMG_7532.JPG   left    white  45.773014


In [29]:
print("\nMean error:")
print(
    errors_df.groupby("keypoint")["error_px"]
    .mean()
)


Mean error:
keypoint
inner    26.457336
outer    24.978112
white    25.888079
Name: error_px, dtype: float64


In [30]:
print("\nOverall mean error:")

print(
    errors_df["error_px"].mean()
)


Overall mean error:
25.77450903256734


In [32]:
import matplotlib.pyplot as plt
import cv2
import numpy as np
from pathlib import Path

for result in test_results:

    image_name = Path(result.path).name

    img = cv2.imread(result.path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    h, w = img.shape[:2]

    plt.figure(figsize=(12, 12))
    plt.imshow(img)

    # -----------------------------
    # Ground Truth
    # -----------------------------

    gt = landmarks[image_name]

    gt_eyes = [
        [
            gt["left_eye_inner_corner"],
            gt["left_eye_outer_corner"],
            gt["left_white_point"]
        ],
        [
            gt["right_eye_inner_corner"],
            gt["right_eye_outer_corner"],
            gt["right_white_point"]
        ]
    ]

    # Plot GT
    for eye_idx, eye in enumerate(gt_eyes):

        for kp_idx, (x, y) in enumerate(eye):

            plt.scatter(
                x, y,
                s=80,
                marker="o"
            )

            plt.text(
                x + 10,
                y + 10,
                f"GT {eye_idx}-{KEYPOINT_ORDER[kp_idx]}",
                fontsize=8
            )

    # -----------------------------
    # Predictions
    # -----------------------------

    if result.keypoints is not None:

        pred_points = result.keypoints.xy.cpu().numpy()

        for eye_idx, eye in enumerate(pred_points):

            for kp_idx, (x, y) in enumerate(eye):

                plt.scatter(
                    x, y,
                    s=100,
                    marker="x"
                )

                plt.text(
                    x + 10,
                    y - 10,
                    f"Pred {eye_idx}-{KEYPOINT_ORDER[kp_idx]}",
                    fontsize=8
                )

    plt.title(
        f"{image_name}\n"
        "Circle = Ground Truth | X = Prediction"
    )

    plt.axis("off")
    plt.show()

<Figure size 1200x1200 with 1 Axes>

<Figure size 1200x1200 with 1 Axes>

<Figure size 1200x1200 with 1 Axes>

<Figure size 1200x1200 with 1 Axes>